In [1]:
import model.create_search_index
import pandas as pd
import numpy as np
from main import SearchSystem

In [ ]:
embed_model_small = "intfloat/multilingual-e5-small"
embed_model_base = "intfloat/multilingual-e5-base"
embed_model_large = "Alibaba-NLP/gte-multilingual-base"




#Проекция данных на 2d

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap
import hdbscan
import warnings
warnings.filterwarnings('ignore')

# 1. Загрузка эмбеддингов
# Предположим, файл имеет формат .npy (массив numpy)
# Если у вас другой формат (csv, txt), раскомментируйте соответствующий блок
def load_embeddings(file_path):
    # Пример для .npy
    embeddings = np.load(file_path)
    print(f"Загружено {embeddings.shape[0]} эмбеддингов размерностью {embeddings.shape[1]}")
    return embeddings

# Альтернативно для CSV (без заголовка, каждая строка - вектор)
# import pandas as pd
# df = pd.read_csv(file_path, header=None)
# embeddings = df.values

file_path = "../models/embeddings/intfloat_multilingual-e5-small.npy"  # укажите ваш путь
embeddings = load_embeddings(file_path)

# 2. Применение UMAP для снижения размерности до 2D (или 3D)
# Параметры можно подбирать: n_neighbors, min_dist
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding_2d = reducer.fit_transform(embeddings)
print("Размерность после UMAP:", embedding_2d.shape)

# 3. Визуализация проекции UMAP (базовая)
plt.figure(figsize=(10, 8))
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], s=5, alpha=0.6)
plt.title("UMAP проекция эмбеддингов")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.show()

# 4. Оценка количества кластеров
# Можно использовать несколько методов:

# 4.1. HDBSCAN на UMAP-проекции (автоматическое определение кластеров)
clusterer = hdbscan.HDBSCAN(min_cluster_size=10, gen_min_span_tree=True)
cluster_labels = clusterer.fit_predict(embedding_2d)
n_clusters_hdbscan = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
print(f"Число кластеров по HDBSCAN: {n_clusters_hdbscan} (шумовые точки помечены -1)")

# Визуализация с раскраской по кластерам HDBSCAN
plt.figure(figsize=(10, 8))
palette = sns.color_palette('deep', n_clusters_hdbscan)
colors = [palette[x] if x != -1 else (0.5,0.5,0.5) for x in cluster_labels]
plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=colors, s=5, alpha=0.7)
plt.title(f"Кластеризация HDBSCAN (кластеров: {n_clusters_hdbscan})")
plt.show()

# 4.2. Метод локтя (Elbow) для KMeans на исходных эмбеддингах
# (или на UMAP, но лучше на исходных для оценки структуры)
inertia = []
silhouette_scores = []
K_range = range(2, 30)  # примерный диапазон, можно расширить

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(embeddings)
    inertia.append(kmeans.inertia_)
    # силуэт считаем на подвыборке, если данных много (>10000)
    if len(embeddings) > 10000:
        sample_idx = np.random.choice(len(embeddings), size=10000, replace=False)
        sil = silhouette_score(embeddings[sample_idx], labels[sample_idx])
    else:
        sil = silhouette_score(embeddings, labels)
    silhouette_scores.append(sil)

# Построение графиков для выбора k
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(K_range, inertia, 'bo-')
ax1.set_xlabel('Количество кластеров k')
ax1.set_ylabel('Интерция (сумма квадратов расстояний)')
ax1.set_title('Метод локтя')

ax2.plot(K_range, silhouette_scores, 'ro-')
ax2.set_xlabel('Количество кластеров k')
ax2.set_ylabel('Силуэт')
ax2.set_title('Оценка силуэта')
plt.tight_layout()
plt.show()

# 4.3. Анализ результатов и рекомендация
print("\nРекомендации по выбору количества кластеров для FAISS IVF:")
print("- HDBSCAN дал {} кластеров (не учитывая шум)".format(n_clusters_hdbscan))
print("- По методу локтя и силуэту оптимальное k может находиться в районе излома графика.")
print("\nДля FAISS IVF число кластеров обычно выбирают как компромисс между точностью и скоростью.")
print("Типичные значения: nlist = 4*sqrt(N) до 16*sqrt(N), где N - число векторов.")
print(f"Для вашего N={len(embeddings)} это примерно от {int(4*np.sqrt(len(embeddings)))} до {int(16*np.sqrt(len(embeddings)))}.")
print("Рекомендуется взять число, близкое к полученному из анализа (например, {}), но в пределах разумного для FAISS (обычно 100-1000).".format(n_clusters_hdbscan))

Загружено 244484 эмбеддингов размерностью 384
